In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


df = pd.read_csv(r"C:\Users\rajan\Downloads\cleaned_credit_data_v1.csv")

# Defining features and target
X = df.drop(columns=["default"])
y = df["default"]

# Spliting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Trains the Gaussian Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)


y_pred = nb_model.predict(X_test_scaled)
y_prob = nb_model.predict_proba(X_test_scaled)[:, 1]


# Evaluating the default threshold (0.50)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("Naive Bayes Results")
print("=" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Evaluating with an adjusted threshold

threshold = 0.35
y_pred_adjusted = (y_prob >= threshold).astype(int)

adjusted_precision = precision_score(y_test, y_pred_adjusted, zero_division=0)
adjusted_recall = recall_score(y_test, y_pred_adjusted, zero_division=0)
adjusted_f1 = f1_score(y_test, y_pred_adjusted, zero_division=0)
adjusted_cm = confusion_matrix(y_test, y_pred_adjusted)

print(f"\nAdjusted Threshold Results (threshold = {threshold})")
print("=" * 40)
print(f"Precision: {adjusted_precision:.4f}")
print(f"Recall   : {adjusted_recall:.4f}")
print(f"F1 Score : {adjusted_f1:.4f}")

print("\nAdjusted Confusion Matrix:")
print(adjusted_cm)

#  Cross-validated ROC-AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_scaled_full = scaler.fit_transform(X)

cv_auc_scores = cross_val_score(
    GaussianNB(),
    X_scaled_full,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("\nCross-Validated ROC-AUC")
print("=" * 40)
print("Fold Scores:", cv_auc_scores)
print(f"Mean ROC-AUC: {cv_auc_scores.mean():.4f}")
print(f"Std Dev     : {cv_auc_scores.std():.4f}")

Naive Bayes Results
Accuracy : 0.3544
Precision: 0.2427
Recall   : 0.9042
F1 Score : 0.3826
ROC-AUC  : 0.7339

Confusion Matrix:
[[ 925 3742]
 [ 127 1199]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.20      0.32      4667
           1       0.24      0.90      0.38      1326

    accuracy                           0.35      5993
   macro avg       0.56      0.55      0.35      5993
weighted avg       0.74      0.35      0.34      5993


Adjusted Threshold Results (threshold = 0.35)
Precision: 0.2393
Recall   : 0.9133
F1 Score : 0.3793

Adjusted Confusion Matrix:
[[ 818 3849]
 [ 115 1211]]

Cross-Validated ROC-AUC
Fold Scores: [0.75110294 0.73038796 0.73678916 0.7586016  0.74405222]
Mean ROC-AUC: 0.7442
Std Dev     : 0.0100
